# Columnas desde DXF

Este notebook muestra el uso actualizado de la extracción desde DXF. Los `x_positions` se entregan asociados a una falla específica y las áreas pequeñas se evalúan respecto al área total del DXF.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "dynaengine").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dynaengine import (
    apply_material_aliases,
    extract_columns_from_dxf,
    prepare_column_configs,
    process_column_config,
    resolve_unidentified_materials,
)

## 1. Materiales caracterizados

In [ ]:
MATERIALS = [
    {
        "material_name": "Material de poza",
        "unit_weight_kn_m3": 19,
        "shear_velocity": {"depth": [0, 5, 10, 15], "vs": [300, 350, 440, 550]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "darendeli_2001",
            "sigma_vertical": 100,
            "soil_parameters": {"IP": 0.0, "OCR": 1.0, "k0": 0.7, "frequency": 1.0, "N": 10},
        },
    },
    {
        "material_name": "Material del dique",
        "unit_weight_kn_m3": 20,
        "shear_velocity": {"depth": [0, 8, 15, 20], "vs": [320, 380, 420, 550]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "menq_2003",
            "sigma_vertical": 100,
            "soil_parameters": {"Cu": 18.0, "D50": 8.0, "k0": 0.7, "N": 10},
        },
    },
    {
        "material_name": "Grava arcillosa",
        "unit_weight_kn_m3": 19,
        "shear_velocity": {"depth": [0, 10, 20, 25], "vs": [200, 330, 540, 600]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "wang_2021",
            "sigma_vertical": 100,
            "soil_parameters": {"soil_group": "Nonplastic silty sand group", "Cu": 2.0, "CF": 50.0, "e": 0.7, "D50": 1.0, "wc": 1.0, "k0": 0.7},
        },
    },
    {
        "material_name": "Grava arenosa",
        "unit_weight_kn_m3": 19,
        "shear_velocity": {"depth": [0, 24, 30, 40], "vs": [230, 300, 440, 700]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "rollins_2020",
            "sigma_vertical": 100,
            "soil_parameters": {"Cu": 1.0, "k0": 1.0},
        },
    },
    {
        "material_name": "Grava pobremente gradada",
        "unit_weight_kn_m3": 19.0,
        "shear_velocity": {"depth": [0, 24, 30, 35], "vs": [230, 300, 440, 500]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "ishibashi_1993",
            "sigma_vertical": 100,
            "soil_parameters": {"IP": 50.0, "k0": 0.5},
        },
    },
    {
        "material_name": "Estrato no identificado 1",
        "unit_weight_kn_m3": 19.0,
        "shear_velocity": {"depth": [0, 24, 30, 35], "vs": [230, 300, 440, 550]},
        "shear_properties": {"c": 0, "phi": 34},
        "dynamic_model": {
            "model_type": "rojas_2019",
            "sigma_vertical": 100,
            "soil_parameters": {"k0": 1.0},
        },
    },
]

## 2. Parámetros de extracción

`SMALL_AREA_SCALE` es opcional. Si un polígono tiene `area / area_total_DXF < SMALL_AREA_SCALE`, se omite en el procesamiento de la columna y se extienden las capas superior e inferior hasta el centro del intervalo pequeño.

In [ ]:
dxf_path = ROOT / "examples" / "data" / "section_01.dxf"

# Las posiciones X ahora se asocian explícitamente a una superficie de falla.
# Use llaves "failure_1", "failure_2", ... o llaves numéricas 1, 2, ...
X_POSITIONS_BY_FAILURE = {
    "failure_1": [250.0, 480.0],
}

# Umbral opcional de área pequeña: area_poligono / area_total_DXF.
SMALL_AREA_SCALE = 0.01

# Alias opcionales para nombres de material ya identificados en el DXF.
MATERIAL_ALIASES = {
    # "Nombre en DXF": "Nombre caracterizado en MATERIALS",
}

# Acciones para estratos con nombre "Estrato no identificado ...".
# Debe usarse una de dos opciones por estrato no identificado:
# 1) asignar a un material ya caracterizado; o
# 2) caracterizarlo como un material nuevo, opcionalmente con otro nombre.
UNIDENTIFIED_MATERIAL_ACTIONS = {
    # Opción 1: asignar a material caracterizado
    # "Estrato no identificado 1": {"action": "assign", "material": "Grava arenosa"},

    # Opción 2: caracterizar el estrato no identificado como material nuevo
    # "Estrato no identificado 2": {
    #     "action": "characterize",
    #     "name": "Material nuevo",
    #     "properties": {
    #         "unit_weight_kn_m3": 19,
    #         "shear_velocity": {"depth": [0, 10, 20], "vs": [250, 320, 420]},
    #         "shear_properties": {"c": 0, "phi": 34},
    #         "dynamic_model": {
    #             "model_type": "darendeli_2001",
    #             "sigma_vertical": 100,
    #             "soil_parameters": {"IP": 5, "OCR": 1, "k0": 0.7, "frequency": 1, "N": 10},
    #         },
    #     },
    # },
}

## 3. Extraer columnas y resolver estratos no identificados

Si el DXF contiene estratos con nombre `Estrato no identificado ...`, complete `UNIDENTIFIED_MATERIAL_ACTIONS` antes de continuar. No se preparan columnas con materiales sin caracterizar.

In [ ]:
if not dxf_path.exists():
    raise FileNotFoundError(
        f"No existe el DXF de ejemplo: {dxf_path}. Ajuste dxf_path antes de continuar."
    )

extraction = extract_columns_from_dxf(
    dxf_path,
    x_positions=X_POSITIONS_BY_FAILURE,
    material_aliases=MATERIAL_ALIASES,
    failure_types={"failure_1": "tipo_de_falla"},
    small_area_scale=SMALL_AREA_SCALE,
)

materials_for_dxf, unidentified_aliases = resolve_unidentified_materials(
    MATERIALS,
    extraction.unidentified_materials,
    actions=UNIDENTIFIED_MATERIAL_ACTIONS,
    material_aliases=MATERIAL_ALIASES,
)

ALL_MATERIAL_ALIASES = {**MATERIAL_ALIASES, **unidentified_aliases}
columns = apply_material_aliases(extraction.columns, ALL_MATERIAL_ALIASES)

print(f"Materiales encontrados en DXF: {extraction.material_names}")
print(f"Estratos no identificados: {extraction.unidentified_materials}")
print(f"Alias aplicados: {ALL_MATERIAL_ALIASES}")
print(f"Columnas extraídas: {list(columns.keys())}")

pd.DataFrame(extraction.polygon_area_summary).head()

## 4. Preparar y procesar una columna

In [ ]:
configs = prepare_column_configs(columns, materials_for_dxf, target_frequency_hz=25)
first_name = next(iter(configs))
result = process_column_config(configs[first_name], calibrate=False)

print(f"Columna procesada: {first_name}")
display(result.raw)
display(result.discretized.head())